# Steps 4–5: Warping & Coaddition

LSST will image every patch of sky ~800 times over 10 years. Coaddition
stacks these overlapping visits to produce a single deep image per sky region.

**LSST tasks:**
- `lsst.pipe.tasks.makeWarp.MakeWarpTask` — reproject each visit
- `lsst.pipe.tasks.assembleCoadd.AssembleCoaddTask` — stack warps

**Input:** multiple `calexp` exposures  
**Output:** `deepCoadd` (stacked image on a common sky grid)

**Reference:** Bosch et al. (2018) §4.1–4.3

## 4.1 Sky Tiling: Tracts and Patches

The sky is divided into a hierarchical grid:

- **Tract:** A large region (~1.7° × 1.7° for HSC) with a single TAN projection
- **Patch:** A subdivision of a tract (~4100 × 4100 pixels) — the unit of coadd processing

```
┌─────────────────────────────────────────┐
│              Tract 9813                  │
│  ┌─────┬─────┬─────┬─────┬─────┐       │
│  │ 0,0 │ 1,0 │ 2,0 │ 3,0 │ ... │       │
│  ├─────┼─────┼─────┼─────┼─────┤       │
│  │ 0,1 │ 1,1 │ 2,1 │ 3,1 │ ... │ ← patches │
│  ├─────┼─────┼─────┼─────┼─────┤       │
│  │ ... │ ... │ ... │ ... │ ... │       │
│  └─────┴─────┴─────┴─────┴─────┘       │
└─────────────────────────────────────────┘
```

Each `deepCoadd` is identified by `{band, tract, patch}`.

## 4.2 Warping

Each calibrated visit (`calexp`) is **reprojected** (warped) onto the
coadd's sky grid using its WCS. This involves:

1. For each output pixel in the coadd grid, compute its sky position
2. Use the input exposure's WCS to find the corresponding input pixel
3. Interpolate the input pixel value (3rd-order Lanczos kernel)

### Lanczos interpolation
The Lanczos kernel preserves flux and minimizes ringing:
$$L(x) = \text{sinc}(x) \cdot \text{sinc}(x/a), \quad |x| < a$$
where $a = 3$ (Lanczos-3). This is preferred over bilinear or bicubic
because it has better frequency response — critical for preserving
galaxy shapes through the resampling.

### Why warping matters for shapes
- Interpolation smooths the image slightly → must be accounted for in PSF
- The **coadd PSF** at any point is the weighted combination of warped
  input PSFs, not a simple analytic function

## 4.3 Stacking

Warped exposures are combined using **inverse-variance weighted mean**:

$$I_{\text{coadd}} = \frac{\sum_i w_i \cdot I_i}{\sum_i w_i}, \quad w_i = \frac{1}{\sigma_i^2}$$

This is optimal (minimum-variance) for Gaussian noise. Each pixel also
gets a combined variance and mask.

### Artifact rejection: "Safe Clipping"
Bosch et al. (2018) introduced a novel algorithm to reject artifacts
(satellite trails, cosmic rays) while preserving PSF properties:

1. Build a **direct (unclipped) coadd** — optimal but includes artifacts
2. Build a **clipped coadd** — rejects outlier pixels but distorts the PSF
   (because rejecting the best-seeing visit at a star's core would bias it)
3. **Compare** the two: where they differ significantly, the clipped coadd
   found an artifact. Use the clipped coadd only at those locations;
   use the direct coadd everywhere else.

This preserves the linear PSF properties of the direct coadd while
still removing artifacts.

### Depth from coaddition
If $N$ visits each have noise $\sigma$, the coadd noise is:
$$\sigma_{\text{coadd}} = \frac{\sigma}{\sqrt{N}}$$

LSST will reach ~27.5 mag (5σ, point source, r-band) after 10 years.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift, gaussian_filter

rng = np.random.default_rng(42)

In [ ]:
# --- Simulate coaddition of multiple exposures ---

ny, nx = 128, 128
n_visits = 10
yy, xx = np.mgrid[:ny, :nx]

# True sky scene: a few galaxies
true_scene = np.zeros((ny, nx))
gal_positions = [(40, 60), (80, 30), (90, 90), (30, 100), (65, 65)]
for (cy, cx) in gal_positions:
    flux = 10**rng.uniform(2.5, 4)
    sigma = rng.uniform(2, 5)
    e1 = rng.uniform(-0.3, 0.3)
    # Elliptical Gaussian
    Qxx = sigma**2 * (1 + e1)
    Qyy = sigma**2 * (1 - e1)
    true_scene += flux * np.exp(-0.5*((xx-cx)**2/Qxx + (yy-cy)**2/Qyy))

# Generate visits with different PSFs, dithers, and noise
visits = []
variances = []

for i in range(n_visits):
    # Different seeing each visit
    seeing = rng.uniform(2.0, 4.0)  # pixels FWHM
    psf_sigma = seeing / 2.355

    # Convolve true scene with this visit's PSF
    convolved = gaussian_filter(true_scene, sigma=psf_sigma)

    # Sub-pixel dither (random offset)
    dx, dy = rng.uniform(-2, 2, 2)
    shifted = shift(convolved, [dy, dx], order=3)  # Lanczos-like

    # Add noise (varies with seeing: worse seeing = higher sky)
    noise_level = 5 + 3 * (seeing / 2.0)
    noisy = shifted + rng.normal(scale=noise_level, size=(ny, nx))

    visits.append(noisy)
    variances.append(noise_level**2 * np.ones((ny, nx)))

# Inverse-variance weighted coadd
weights = [1.0 / v for v in variances]
sum_weight = np.sum(weights, axis=0)
coadd = np.sum([w * img for w, img in zip(weights, visits)], axis=0) / sum_weight
coadd_variance = 1.0 / sum_weight

# Simple mean for comparison
simple_mean = np.mean(visits, axis=0)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

vmin, vmax = np.percentile(coadd, [1, 99.5])

# Show 3 individual visits
for i, ax in enumerate(axes[0]):
    ax.imshow(visits[i], cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    noise_i = np.sqrt(variances[i][0, 0])
    ax.set_title(f'Visit {i+1} (noise σ={noise_i:.1f})')
    ax.set_xticks([]); ax.set_yticks([])

# Coadd results
axes[1, 0].imshow(simple_mean, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[1, 0].set_title(f'Simple mean ({n_visits} visits)')

axes[1, 1].imshow(coadd, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[1, 1].set_title(f'Inverse-variance coadd ({n_visits} visits)')

axes[1, 2].imshow(np.sqrt(coadd_variance), cmap='magma', origin='lower')
axes[1, 2].set_title('Coadd noise (√variance)')

for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Coaddition: Stacking Multiple Visits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../figures/coaddition_demo.png', dpi=150, bbox_inches='tight')
plt.show()

# Noise improvement
single_noise = np.sqrt(np.median(variances[0]))
coadd_noise = np.sqrt(np.median(coadd_variance))
print(f"Single-visit noise: σ = {single_noise:.1f}")
print(f"Coadd noise:        σ = {coadd_noise:.1f}")
print(f"Improvement:        {single_noise/coadd_noise:.1f}× (expected √{n_visits} = {np.sqrt(n_visits):.1f}×)")

## The Coadd PSF

The coadd PSF is **not** a simple analytic function. At each position,
it is the weighted average of the warped PSFs from every input visit:

$$\text{PSF}_{\text{coadd}}(\vec{x}) = \frac{\sum_i w_i \cdot \text{PSF}_i(\vec{x})}{\sum_i w_i}$$

The pipeline stores this as a `CoaddPsf` object that can be evaluated
at any position. It internally:
1. Finds which input visits overlap the query position
2. Transforms to each visit's coordinate system
3. Evaluates each visit's PSF model
4. Warps back and combines with inverse-variance weights

This is computationally expensive but exact. For shape measurement,
the PSF must be evaluated at every galaxy position.

**Next:** [06_detection_deblending.ipynb](06_detection_deblending.ipynb) — Source detection and deblending